## seed_dim_geo
Seeds `silver.dim_geo` (CBSA grain) from the committed crosswalk reference data on the
`reference` Volume (`CROSSWALKS`): `cbsa_master.csv` (OMB 2023 universe) is the row source;
`zillow_region_to_cbsa.csv` supplies `zillow_region_id`; `household_rank` is enriched from
`bronze.realtor_metro_monthly` (latest month per CBSA).

Idempotent **MERGE on `cbsa_code`** — `geo_key` is GENERATED ALWAYS AS IDENTITY and is
omitted from INSERT (Delta assigns it), and MERGE keeps it **stable** across re-seeds so
fact foreign keys never break. Upload the CSVs to the Volume first (rare reference refresh).

In [ ]:
%run "../libs/notebook_init"

In [ ]:
# notebook_init injects SILVER, BRONZE, CROSSWALKS, F, spark. Window is not injected.
from pyspark.sql import Window

# Reference CSVs from the reference Volume (all STRING; cast household_rank to INT).
cbsa = spark.read.option("header", True).csv(f"{CROSSWALKS}cbsa_master.csv")
xwalk = (
    spark.read.option("header", True).csv(f"{CROSSWALKS}zillow_region_to_cbsa.csv")
    .where("cbsa_code IS NOT NULL AND cbsa_code <> ''")
)

# One zillow_region_id per CBSA (collisions are rare; take the min deterministically).
region_by_cbsa = xwalk.groupBy("cbsa_code").agg(F.min("region_id").alias("zillow_region_id"))

# household_rank = the most recent month's value per CBSA from Bronze Realtor.
realtor = spark.table(f"{BRONZE}.realtor_metro_monthly").where(
    "household_rank IS NOT NULL AND household_rank <> ''"
)
w = Window.partitionBy("cbsa_code").orderBy(F.col("month_date_yyyymm").desc())
hh = (
    realtor.withColumn("rn", F.row_number().over(w)).where("rn = 1")
    .select("cbsa_code", F.col("household_rank").cast("double").cast("int").alias("household_rank"))
)

staging = (
    cbsa.join(region_by_cbsa, "cbsa_code", "left")
        .join(hh, "cbsa_code", "left")
        .select("cbsa_code", "cbsa_title", "cbsa_type", "zillow_region_id",
                "primary_state", "state_list", "household_rank")
)
staging.createOrReplaceTempView("dim_geo_staging")

# geo_key (IDENTITY) is omitted from INSERT so Delta assigns it; MERGE keeps it stable.
spark.sql(f"""
    MERGE INTO {SILVER}.dim_geo t USING dim_geo_staging s ON t.cbsa_code = s.cbsa_code
    WHEN MATCHED THEN UPDATE SET
        t.cbsa_title=s.cbsa_title, t.cbsa_type=s.cbsa_type,
        t.zillow_region_id=s.zillow_region_id, t.primary_state=s.primary_state,
        t.state_list=s.state_list, t.household_rank=s.household_rank,
        t.updated_ts=current_timestamp()
    WHEN NOT MATCHED THEN INSERT
        (cbsa_code, cbsa_title, cbsa_type, zillow_region_id, primary_state, state_list,
         household_rank, inserted_ts, updated_ts)
        VALUES (s.cbsa_code, s.cbsa_title, s.cbsa_type, s.zillow_region_id, s.primary_state,
                s.state_list, s.household_rank, current_timestamp(), current_timestamp())
""")

total = spark.table(f"{SILVER}.dim_geo").count()
with_zillow = spark.table(f"{SILVER}.dim_geo").where("zillow_region_id IS NOT NULL").count()
with_rank = spark.table(f"{SILVER}.dim_geo").where("household_rank IS NOT NULL").count()
print(f"seed_dim_geo: dim_geo rows = {total:,} (zillow_region_id: {with_zillow:,}, "
      f"household_rank: {with_rank:,})")